In [2]:
!pip install urllib3
!pip install beautifulsoup4
!pip install selenium
!pip install chromedriver-autoinstaller
!pip install webdriver-manager

  Using cached selenium-4.27.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached trio-0.27.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached trio_websocket-0.11.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.2.0-py3-none-any.whl.metadata (5.6 kB)
Using cached selenium-4.27.1-py3-none-any.whl (9.7 MB)
Using cached trio-0.27.0-py3-none-any.whl (481 kB)
Using cached trio_websocket-0.11.1-py3-none-any.whl (17 kB)
Using cached wsproto-1.2.0-py3-none-any.whl (24 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
  Using cached chromedriver_autoinstaller-0.6.4-py3-none-any.whl.metadata (2.1 kB)
Using cached chromedriver_autoinstaller-0.6.4-py3-none-any.whl (7.6 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using c

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import pandas as pd
import requests
from datetime import datetime
import time

def google_search_crawler(search_query, n_pages, chrome_driver_path):
    # Selenium WebDriver 초기화
    service = Service(chrome_driver_path)
    driver = webdriver.Chrome(service=service)
    
    # 기본 URL 설정
    base_url = "https://www.google.com/search?q=" + search_query
    driver.get(base_url)
    driver.implicitly_wait(10)
    
    results = []
    
    # 페이지 순회
    for page in range(n_pages):
        # HTML 파싱
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        # 검색 결과 추출
        search_results = soup.select(".tF2Cxc")
        
        for result in search_results:
            try:
                # URL 추출
                url = result.select_one(".yuRUbf a")['href']
                
                # 제목 추출
                title = result.select_one(".LC20lb").text
                
                # 본문 내용 크롤링
                response = requests.get(url)
                response.encoding = response.apparent_encoding  # 인코딩 설정
                content_soup = BeautifulSoup(response.content, 'html.parser')
                
                # 본문 내용 추출 (필터링 추가)
                paragraphs = content_soup.find_all('p', {'data-ke-size': 'size16'})
                content = "\n".join([p.get_text(separator="\n", strip=True) for p in paragraphs if p.get_text().strip()])  # 빈 텍스트 필터링
                
                # 제외할 내용 필터링 (예: 이미지 관련 텍스트 등)
                content = "\n".join([line for line in content.split("\n") if "사진출처" not in line and len(line) > 20])  # 사진출처 및 짧은 텍스트 제외
                
                # 날짜 정보 추출
                try:
                    date = datetime.now().strftime("%Y-%m-%d")  # 현재 날짜 추가
                except Exception as e:
                    date = "날짜 정보를 찾을 수 없습니다."
                
                # 결과 저장
                results.append({
                    "url": url,
                    "title": title,
                    "content": content,
                    "date": date
                })
            except Exception as e:
                print(f"Error processing result: {e}")
        
        # 다음 페이지로 이동
        try:
            next_button = driver.find_element(By.LINK_TEXT, "다음")
            next_button.click()
            time.sleep(2)  # 페이지 로드 대기
        except Exception as e:
            print("다음 페이지가 없습니다. 크롤링 종료.")
            break

    # WebDriver 종료
    driver.quit()
    
    # DataFrame 생성
    df = pd.DataFrame(results)
    return df


In [5]:
import re

def google_search_crawler(search_query, n_pages, chrome_driver_path):
    # Selenium WebDriver 초기화
    service = Service(chrome_driver_path)
    driver = webdriver.Chrome(service=service)
    
    # 기본 URL 설정
    base_url = "https://www.google.com/search?q=" + search_query
    driver.get(base_url)
    driver.implicitly_wait(10)
    
    results = []
    
    # 페이지 순회
    for page in range(n_pages):
        # HTML 파싱
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        # 검색 결과 추출
        search_results = soup.select(".tF2Cxc")
        
        for result in search_results:
            try:
                # URL 추출
                url = result.select_one(".yuRUbf a")['href']
                
                # 제목 추출
                title = result.select_one(".LC20lb").text
                
                # 본문 내용 크롤링
                response = requests.get(url)
                response.encoding = response.apparent_encoding  # 인코딩 설정
                content_soup = BeautifulSoup(response.content, 'html.parser')
                
                # 본문 내용 추출 (필터링 추가)
                paragraphs = content_soup.find_all('p', {'data-ke-size': 'size16'})
                content = "\n".join([p.get_text(separator="\n", strip=True) for p in paragraphs if p.get_text().strip()])  # 빈 텍스트 필터링
                
                # URL 제거
                content = re.sub(r'http[s]?://\S+', '', content)
                
                # 제외할 내용 필터링 (예: 이미지 관련 텍스트 등)
                content = "\n".join([line for line in content.split("\n") if "사진출처" not in line and len(line) > 20])  # 사진출처 및 짧은 텍스트 제외
                
                # 날짜 정보 추출
                try:
                    date = datetime.now().strftime("%Y-%m-%d")  # 현재 날짜 추가
                except Exception as e:
                    date = "날짜 정보를 찾을 수 없습니다."
                
                # 결과 저장
                results.append({
                    "url": url,
                    "title": title,
                    "content": content,
                    "date": date
                })
            except Exception as e:
                print(f"Error processing result: {e}")
        
        # 다음 페이지로 이동
        try:
            next_button = driver.find_element(By.LINK_TEXT, "다음")
            next_button.click()
            time.sleep(2)  # 페이지 로드 대기
        except Exception as e:
            print("다음 페이지가 없습니다. 크롤링 종료.")
            break

    # WebDriver 종료
    driver.quit()
    
    # DataFrame 생성
    df = pd.DataFrame(results)
    return df


In [12]:
# search_query: 검색어 입력
# n_pages: if문 실행 후 페이지수 입력
# 다이슨 무선청소기는 9 입력
# chrome_driver_path는 Chromedriver.exe 다운 경로
# 크롬 버전이랑 맞는 크롬드라이버.exe 다운 받아서 그냥 C드라이브에 대충 아래처럼 넣기기
if __name__ == "__main__":
    # 사용자 입력 설정
    search_query = "로봇청소기 site:tistory.com"  # 검색어
    n_pages = int(input("크롤링할 페이지 수를 입력하세요: "))  # 페이지 수 입력
    chrome_driver_path = "C:/chromedriver_win64/chromedriver/chromedriver.exe"  # Chromedriver 경로
    
    # 크롤링 실행
    tistory_robot_script = google_search_crawler(search_query, n_pages, chrome_driver_path)
    
    # 결과 출력 및 저장
    print(tistory_robot_script)
    tistory_robot_script.to_csv("tistory_robot_script.csv", index=False, encoding='utf-8-sig')


다음 페이지가 없습니다. 크롤링 종료.
                                                   url  \
0                         https://alsk-1.tistory.com/8   
1                  https://elecinsight.tistory.com/540   
2    https://rnrnrn13455.tistory.com/entry/%EB%A1%9...   
3              https://mistarleehaebom.tistory.com/103   
4                    https://sodolsodan.tistory.com/22   
..                                                 ...   
265                     https://wissm10.tistory.com/36   
266                      https://hunss2.tistory.com/69   
267                    https://xinews.tistory.com/6479   
268                    https://e9e9-9999.tistory.com/3   
269  https://delmoheart.tistory.com/entry/LG-%EB%A1...   

                                         title  \
0        로봇청소기 구매과정 및 후기와 찝찝한 것 - 클리엔R9 - 이것저것   
1       삼성 로봇청소기 VR7MD97716G vs VR7MD96516G 비교   
2       로봇 청소기 추천 구매 시 고려해야 할 3가지 - 구구구 - 티스토리   
3                물걸레 청소에 특화된 필립스 로봇청소기를 소개합니다.   
4                  로봇청소기 추천! 로보

## 기간 별 ##

In [2]:
import re
import pandas as pd
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import requests

def google_search_crawler(search_query, chrome_driver_path, start_year, end_year):
    service = Service(chrome_driver_path)
    driver = webdriver.Chrome(service=service)
    
    results = []
    
    for year in range(start_year, end_year + 1):
        base_url = f"https://www.google.com/search?q={search_query}&tbs=cdr:1,cd_min:1/1/{year},cd_max:12/31/{year}"
        driver.get(base_url)
        driver.implicitly_wait(10)
        
        while True:  # 마지막 페이지까지 자동으로 반복
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            search_results = soup.select(".tF2Cxc")
            
            for result in search_results:
                try:
                    url = result.select_one(".yuRUbf a")['href']
                    title = result.select_one(".LC20lb").text
                    response = requests.get(url)
                    response.encoding = response.apparent_encoding
                    content_soup = BeautifulSoup(response.content, 'html.parser')
                    paragraphs = content_soup.find_all('p')
                    content = "\n".join([p.get_text(strip=True) for p in paragraphs if len(p.get_text().strip()) > 20])
                    content = re.sub(r'http[s]?://\S+', '', content)
                    date = datetime.now().strftime("%Y-%m-%d")
                    
                    results.append({
                        "url": url,
                        "title": title,
                        "content": content,
                        "date": date
                    })
                except Exception as e:
                    print(f"Error processing result: {e}")
            
            try:
                next_button = driver.find_element(By.LINK_TEXT, "다음")
                next_button.click()
                time.sleep(2)
            except Exception:
                print(f"{year}년 검색 크롤링 완료.")
                break

    driver.quit()
    df = pd.DataFrame(results)
    return df


In [11]:

if __name__ == "__main__":
    search_query = "AI에어드레서 site:tistory.com"  # 검색어
    chrome_driver_path = "C:/chromedriver_win64/chromedriver/chromedriver.exe"  # Chromedriver 경로
    start_year = 2020  # 시작 연도
    end_year = 2024  # 종료 연도
    
    tistory_robot_script = google_search_crawler(search_query, chrome_driver_path, start_year, end_year)
    
    print(tistory_robot_script)
    tistory_robot_script.to_csv("tistory_samsung_airdresser_ai_year.csv", index=False, encoding='utf-8-sig')

2020년 검색 크롤링 완료.
2021년 검색 크롤링 완료.
2022년 검색 크롤링 완료.
2023년 검색 크롤링 완료.
2024년 검색 크롤링 완료.
                                                  url  \
0   https://the-god-of-charge.tistory.com/category...   
1                        https://kt99.tistory.com/385   
2                        https://kt99.tistory.com/355   
3                   https://prizeking.tistory.com/835   
4                        https://kt99.tistory.com/354   
..                                                ...   
87                  https://realtor87.tistory.com/253   
88  https://tiguin.tistory.com/entry/%EC%82%BC%EC%...   
89  https://lazion.tistory.com/category/%23%EA%B0%...   
90  https://beyondthebook.tistory.com/entry/%EA%B0...   
91                     https://godsbe.tistory.com/tag   

                                      title  \
0     '분류 전체보기' 카테고리의 글 목록 (48 Page) - 오-나라   
1                         남양주 지역화폐 가맹점 알아보기   
2                평택시 지역화폐 가맹점 알아보기 - 그냥사는거지   
3         상하수도 관련주 & 상하수도 주식 총정리 - 경제와 유아독